# Citation Provenance

nidus exposes citations as first-class objects. This notebook
explores the citation graph: which papers support which
parameters, which papers are most load-bearing, and how to
export the whole bibliography as BibTeX for use in a paper or
thesis.

## Setup

In [ ]:
import nidus
from collections import Counter

ds = nidus.load()
print(f"{len(ds.citations)} citations indexed")

## Most load-bearing citations

How many parameters in the dataset trace back to each paper?
The top of this list shows which sources the dataset is most
sensitive to — if any of these were retracted, a large fraction
of nidus would need re-tiering.

In [ ]:
uses = Counter(c.key for p in ds for c in p.citations)
for key, n in uses.most_common(8):
    c = ds.citations[key]
    authors = ", ".join(c.authors[:2]) + (" et al." if len(c.authors) > 2 else "")
    print(f"  {n:>2} parameters  ·  {authors} ({c.year}) — {c.title[:70]}")

## Inspect a single citation

In [ ]:
c = ds.citations["mahendru-2014-cardiac-output"]
print(f"Title:    {c.title}")
print(f"Authors:  {', '.join(c.authors)}")
print(f"Year:     {c.year}")
print(f"Journal:  {c.journal}")
print(f"DOI:      {c.doi}")
print()
print("Supports these parameters:")
for p in ds.citations_for(c.key):
    primary = " (primary)" if p.primary_citation and p.primary_citation.key == c.key else ""
    print(f"  · [{p.tier}] {p.id}{primary}")

## Citations without a resolvable identifier

Most citations have a DOI or PMID. Older book references
predate DOIs and are kept in the dataset for accuracy. The
schema accepts them without an identifier; the weekly
reachability check skips them rather than failing.

In [ ]:
unresolved = [c for c in ds.citations.values() if not (c.doi or c.pmid or c.url)]
for c in unresolved:
    print(f"  {c.key}  ({c.type})  — {c.title}")
if not unresolved:
    print("  (none — every citation has at least one of DOI / PMID / URL)")

## Export the whole bibliography as BibTeX

In [ ]:
_BIBTEX_TYPE_MAP = {
    "journal-article": "article",
    "book": "book",
    "book-chapter": "incollection",
    "conference-paper": "inproceedings",
    "preprint": "misc",
    "report": "techreport",
    "dataset": "misc",
    "thesis": "phdthesis",
}

def to_bibtex(c):
    entry_type = _BIBTEX_TYPE_MAP.get(c.type, "misc")
    fields = []
    def add(name, value):
        if value not in (None, "", []):
            fields.append(f"  {name} = {{{value}}}")
    add("title", c.title)
    add("author", " and ".join(c.authors))
    add("year", c.year)
    if entry_type == "article":
        add("journal", c.journal)
    else:
        add("publisher", c.publisher or c.journal)
    add("volume", c.volume)
    add("number", c.issue)
    add("pages", c.pages)
    add("doi", c.doi)
    add("pmid", c.pmid)
    add("url", c.url)
    add("isbn", c.isbn)
    return f"@{entry_type}{{{c.key},\n" + ",\n".join(fields) + "\n}}"

print(to_bibtex(ds.citations["mahendru-2014-cardiac-output"]))

In [ ]:
# Full bibliography (truncated for display):
bib = "\n\n".join(to_bibtex(c) for c in ds.citations.values())
print(bib[:600] + "\n...")
print(f"\n[full bibliography is {len(bib):,} characters across {len(ds.citations)} entries]")

The dashboard's Download page exposes the same BibTeX export as
a single-click download. Both rely on the citation metadata
encoded in ``dataset/citations/citations.json``.